# Modul 11: Deteksi Anomali Smart Meter (Isolation Forest & LSTM Autoencoder)
**Mata Kuliah:** Analisis Deret Waktu dan Peramalan (TEE-412) — S1 Teknik Elektro FT UNIB  
**Sub-CPMK 12 (Minggu 13):** Merancang detektor anomali tanpa pengawasan (*unsupervised*) berbasis galat rekonstruksi Autoencoder ($\tau = \mu + 3\sigma$) dan mengevaluasinya pada benchmark sintetis serta dataset terkurasi `smartmeter_anomaly_168h.csv`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import torch
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)

## 1. Benchmark Profil Konsumsi Harian 24-Jam Smart Meter (Normal vs. Pencurian Listrik)

In [ ]:
hours = np.arange(24)
base_profile = 3.5 + 1.8 * np.sin(2 * np.pi * (hours - 7) / 24) + 1.5 * np.exp(-((hours - 19) ** 2) / 6)

# Data latih HANYA berisi profil normal (unsupervised)
n_train = 400
X_train = np.array([base_profile + np.random.normal(0, 0.25, 24) for _ in range(n_train)], dtype=np.float32)

# Data uji: 180 normal + 20 anomali pencurian/bypass (10% imbalance)
X_test_norm = np.array([base_profile + np.random.normal(0, 0.25, 24) for _ in range(180)], dtype=np.float32)
X_test_anom = np.array([base_profile * 0.35 + np.random.normal(0, 0.15, 24) for _ in range(20)], dtype=np.float32)
X_test = np.vstack([X_test_norm, X_test_anom])
y_true = np.array([0] * 180 + [1] * 20)  # 0=Normal, 1=Anomali

## 2. Pelatihan LSTM Autoencoder & Penentuan Ambang $\tau = \mu + 3\sigma$

In [ ]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, hidden_dim=16):
        super().__init__()
        self.encoder = nn.LSTM(1, hidden_dim, batch_first=True)
        self.decoder = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.out_proj = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        _, (h, _) = self.encoder(x)
        repeated = h.repeat(x.size(1), 1, 1).permute(1, 0, 2)
        dec_out, _ = self.decoder(repeated)
        return self.out_proj(dec_out)

ae = LSTMAutoencoder(hidden_dim=16)
opt = torch.optim.Adam(ae.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

X_tr_t = torch.from_numpy(X_train).unsqueeze(-1)
X_te_t = torch.from_numpy(X_test).unsqueeze(-1)

for epoch in range(60):
    opt.zero_grad()
    loss = loss_fn(ae(X_tr_t), X_tr_t)
    loss.backward()
    opt.step()

ae.eval()
with torch.no_grad():
    tr_recon = ae(X_tr_t).numpy().squeeze(-1)
    te_recon = ae(X_te_t).numpy().squeeze(-1)

tr_mse = np.mean((X_train - tr_recon) ** 2, axis=1)
te_mse = np.mean((X_test - te_recon) ** 2, axis=1)

threshold = tr_mse.mean() + 3.0 * tr_mse.std()
y_pred_ae = (te_mse > threshold).astype(int)

# Bandingkan dengan Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42).fit(X_train)
y_pred_iso = (iso.predict(X_test) == -1).astype(int)

for name, yp, score in [('LSTM Autoencoder', y_pred_ae, te_mse), ('Isolation Forest', y_pred_iso, -iso.score_samples(X_test))]:
    print(f"=== {name} (Benchmark 24h) ===")
    print(f"  Precision={precision_score(y_true, yp):.3f} | Recall={recall_score(y_true, yp):.3f} | F1={f1_score(y_true, yp):.3f} | AUROC={roc_auc_score(y_true, score):.3f}")
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, yp)}")

## 3. Uji Deteksi pada Dataset Terkurasi `smartmeter_anomaly_168h.csv`
Dataset 168 jam ini memuat **1 point anomaly** ($t=42$) dan **1 collective anomaly / bypass meter** ($t=110\text{–}128$).

In [ ]:
CSV_NAME = 'smartmeter_anomaly_168h.csv'
RAW_URL = 'https://raw.githubusercontent.com/novatha/deret-waktu-tee412/main/labs/datasets/' + CSV_NAME

def load_smartmeter():
    for p in [Path('datasets') / CSV_NAME, Path('labs/datasets') / CSV_NAME]:
        if p.exists():
            print(f"Memuat dataset lokal: {p}")
            return pd.read_csv(p, parse_dates=['Timestamp'], index_col='Timestamp')
    try:
        print(f"Mengunduh dataset dari GitHub: {RAW_URL}")
        return pd.read_csv(RAW_URL, parse_dates=['Timestamp'], index_col='Timestamp')
    except Exception as e:
        print(f"Fallback sintetis (alasan: {e})")
        t = np.arange(168)
        idx = pd.date_range('2026-09-07', periods=168, freq='h')
        p_kw = 2.8 + 1.2 * np.sin(2 * np.pi * (t - 7) / 24) + np.random.normal(0, 0.15, 168)
        v_v = 222 + np.random.normal(0, 1.2, 168)
        lbl = np.zeros(168, dtype=int)
        p_kw[42] += 6.5; lbl[42] = 1
        p_kw[110:129] *= 0.25; v_v[110:129] -= 12; lbl[110:129] = 1
        return pd.DataFrame({'Power_kW': p_kw, 'Voltage_V': v_v, 'Is_Anomaly_Label': lbl}, index=idx)

df_sm = load_smartmeter()
feats = df_sm[['Power_kW', 'Voltage_V']].copy()
feats['hour_sin'] = np.sin(2 * np.pi * df_sm.index.hour / 24.0)
feats['hour_cos'] = np.cos(2 * np.pi * df_sm.index.hour / 24.0)
X_sm = StandardScaler().fit_transform(feats)

iso_sm = IsolationForest(contamination=0.12, random_state=42).fit(X_sm)
scores_sm = -iso_sm.score_samples(X_sm)
pred_sm = (iso_sm.predict(X_sm) == -1).astype(int)
y_sm = df_sm['Is_Anomaly_Label'].to_numpy()

print(f"=== Evaluasi smartmeter_anomaly_168h.csv (Isolation Forest) ===")
print(f"  Precision={precision_score(y_sm, pred_sm):.3f} | Recall={recall_score(y_sm, pred_sm):.3f} | F1={f1_score(y_sm, pred_sm):.3f} | AUROC={roc_auc_score(y_sm, scores_sm):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(te_mse[y_true == 0], bins=25, alpha=0.7, label='Normal', color='steelblue')
axes[0].hist(te_mse[y_true == 1], bins=15, alpha=0.7, label='Anomali Pencurian', color='crimson')
axes[0].axvline(threshold, color='black', ls='--', lw=2, label=f'Ambang tau = {threshold:.3f}')
axes[0].set_title('Distribusi Galat Rekonstruksi LSTM Autoencoder')
axes[0].set_xlabel('MSE Rekonstruksi')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(df_sm.index, df_sm['Power_kW'], color='navy', lw=1.4, label='Power_kW')
anom_idx = df_sm.index[pred_sm == 1]
axes[1].scatter(anom_idx, df_sm.loc[anom_idx, 'Power_kW'], color='crimson', s=35, zorder=5, label='Terdeteksi Anomali')
axes[1].set_title('Deteksi Anomali pada smartmeter_anomaly_168h.csv')
axes[1].set_ylabel('Daya (kW)')
axes[1].legend()
axes[1].grid(True)
plt.tight_layout()
plt.show()

## 4. Tugas & Pertanyaan Analisis (C4–C6)
1. **[C4]** Bandingkan performa deteksi terhadap *point anomaly* ($t=42$) vs. *collective anomaly/bypass* ($t=110\text{–}128$) pada `smartmeter_anomaly_168h.csv`.
2. **[C5]** Lakukan sapuan ambang $k \in \{1.5, 2.0, 2.5, 3.0, 3.5\}$ pada $\tau = \mu + k\sigma$ dan plot *trade-off* Precision vs. Recall!